# 샤프트 외경 측정 — YOLO 모델 비교 (팀 공용 Colab)

## 사용 순서

1. Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
2. 위에서부터 셀을 순서대로 실행합니다.
3. **[내 설정] 셀의 `MODEL` 한 줄만** 본인 배정 모델로 바꿉니다.
4. 마지막 학습 셀을 실행합니다. 결과와 체크포인트는 공유 Drive에 저장되며, 중단 후 재실행하면 `last.pt`에서 이어집니다.

`dataset/`은 모두가 읽기만 합니다. 모델별 `runs/{model}/`, `results/result_{model}.json`만 작성됩니다.


In [ ]:
# 1) 패키지 설치 — 처음 한 번 실행
!pip install -q -U ultralytics
ULTRALYTICS_READY = True
print('ultralytics 설치 완료')


In [ ]:
# 2) Google Drive 연결 — 권한 허용 팝업에서 본인 계정을 선택하세요
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) GPU 확인 — Tesla T4 등이 표시되면 정상
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾼 뒤 이 셀부터 다시 실행하세요.')


## 4) 내 설정 — `MODEL` 한 줄만 변경

가능한 값: `yolo26n.pt`, `yolo26s.pt`, `yolo26m.pt`, `yolo11n.pt`, `yolo11s.pt`, `yolo11m.pt`


In [ ]:
# ══════════ [내 설정] 여기 MODEL 한 줄만 바꾸세요 ══════════
MODEL = 'yolo26n.pt'     # ★ 내 배정 모델
# ═══════════════════════════════════════════════════════
DRIVE = '/content/drive/MyDrive/shaft_sweep'


## 5) 학습·평가 — 위 설정을 확인한 뒤 이 셀 하나를 실행하세요

공유 데이터셋 검사 → 시편 단위 split → 학습/resume → 탐지 평가 → 모델별 캘리브레이션 → 측정 평가 → JSON·CSV 저장을 순서대로 수행합니다.


In [ ]:
"""
================================================================
 샤프트 외경 측정 — YOLO 모델 비교 팀 공통 베이스 코드
================================================================
 하는 일: 공유 데이터셋 읽기 → split(자동) → 내 모델 학습 → 탐지·측정 평가 → 결과 JSON 저장
 사용법 : 맨 위 [설정]에서 MODEL 한 줄만 내가 배정받은 것으로 바꾸고 실행!
 ★ 학습 설정(epochs/imgsz/batch/seed)과 측정 규격은 팀 전원 동일해야 비교가 공정합니다 — 건드리지 마세요.

 전원 공통 조건
 ┌──────────────┬──────────────────────────────────────────────────────────┐
 │ 데이터셋     │ MyDrive/shaft_sweep/dataset/ (A가 만든 것, 수정 금지)   │
 │ 이미지       │ 가로 2992 px                                             │
 │ split        │ SEED=42, 시편 단위 8:2, specimen_map.csv 기준            │
 │ 학습         │ 100 epochs, 640, batch 16, AdamW, lr0 .001, patience 20  │
 │ 증강         │ mosaic .5, flip/degrees/shear/perspective 0, scale .2    │
 │ 측정         │ 18~82%, 31점, median, sub-pixel, MAD 3σ (D-003)          │
 │ 캘리브레이션 │ 방법은 동일, 값은 모델마다 다시 산출                     │
 └──────────────┴──────────────────────────────────────────────────────────┘

 Limitations
 - 라벨은 기존 best.pt의 pre-annotation을 사람이 검수한 것이다. 따라서 이 실험은
   “어느 아키텍처가 기존 박스 규칙을 가장 정밀하게 재현하는가”를 비교한다.
 - 표본은 시편 13개/이미지 161장으로 작다. 결과 해석 시 표본 수를 함께 제시한다.
 - mAP 1등과 측정 재현성 1등이 다르면 측정 재현성(repeat_sigma_um)을 우선한다.
"""

# ══════════ [설정] 여기 한 줄만 바꾸세요 ══════════
# MODEL은 위 [내 설정] 셀에서 정의합니다.
#   yolo26n.pt / yolo26s.pt / yolo26m.pt / yolo11n.pt / yolo11s.pt / yolo11m.pt

# DRIVE는 위 [내 설정] 셀에서 정의합니다.
# ────────── 아래는 전원 동일, 건드리지 마세요 ──────────
EPOCHS, IMGSZ, BATCH, SEED = 100, 640, 16, 42
RATIO = (0.8, 0.2)        # train / val (시편 단위 분할)

# D-003 측정 규격 — 희연님 measurement_engine.py V2와 동일
CONF, ROI_VERTICAL_MARGIN = 0.25, 0.40
X_START, X_END, N_LINES = 0.18, 0.82, 31
UPPER_END, LOWER_START = 0.46, 0.54
MIN_DIAMETER, MAX_DIAMETER = 0.25, 0.60
GRAD_PERCENTILE, STRENGTH_RATIO = 82, 0.55
MIN_RAW_EDGES, MIN_VALID_EDGES, MAD_K = 10, 8, 3.0
CALIBRATION_MM = 20.021        # ★ calib 시편의 실측 외경(mm). calib 이미지를 바꾸면 이 값도 반드시 같이 바꾸세요.
MIN_CALIB_IMAGES = 3           # 캘리브레이션 median이 의미를 가지려면 최소 3장
CALIB_SPREAD_MAX_PCT = 2.0     # calib 이미지끼리 측정 픽셀이 이 % 이상 벌어지면 중단
CALIB_SCALE_MAX_PCT = 10.0     # calib 픽셀이 본 이미지 측정 픽셀 중앙값과 이 % 이상 다르면 중단
SPEC_LOWER, SPEC_UPPER = 20.010, 20.030
ALLOWED_MODELS = {
    "yolo26n.pt", "yolo26s.pt", "yolo26m.pt",
    "yolo11n.pt", "yolo11s.pt", "yolo11m.pt",
}
# ═══════════════════════════════════════════════

from collections import Counter
from datetime import datetime, timezone
import csv
import getpass
import json
import logging
import math
import os
from pathlib import Path
import random
import subprocess
import sys
import time
import traceback


def stop(message: str, code: int = 1) -> None:
    """긴 traceback 대신, 팀원이 바로 행동할 수 있는 한국어 안내를 보여준다."""
    print(f"\n[중단] {message}\n")
    raise SystemExit(code)


def prepare_environment():
    print("[1/7] 환경을 준비합니다.")
    # Colab 노트북의 패키지 셀을 이미 실행했으면 같은 설치를 반복하지 않는다.
    if not globals().get("ULTRALYTICS_READY", False):
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"],
                check=True,
            )
        except subprocess.CalledProcessError:
            stop("ultralytics 설치/업데이트에 실패했습니다. 인터넷 연결 후 다시 실행하세요.")
    else:
        print("  ultralytics 설치 완료 상태를 사용합니다.")

    try:
        import cv2
        import numpy as np
        import pandas as pd
        import torch
        import ultralytics
        from ultralytics import YOLO
    except ImportError as exc:
        stop(f"필수 패키지를 불러오지 못했습니다: {exc}. 런타임을 재시작한 뒤 다시 실행하세요.")

    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except ImportError:
        pass
    except Exception as exc:
        stop(f"Google Drive 마운트에 실패했습니다: {exc}")

    device = 0 if torch.cuda.is_available() else "cpu"
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"  ultralytics={ultralytics.__version__} / 장치={gpu}")
    if device == "cpu":
        print("  [경고] GPU가 없습니다. Colab 메뉴에서 GPU 런타임으로 바꾸는 것을 권장합니다.")
    return cv2, np, pd, torch, ultralytics, YOLO, device, gpu


def image_files(folder: Path):
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and not p.name.startswith("._") and p.suffix.lower() in suffixes
    )


def validate_dataset(root: Path, cv2, pd):
    dataset = root / "dataset"
    images_dir, labels_dir, calib_dir = dataset / "images", dataset / "labels", dataset / "calib"
    map_path = dataset / "specimen_map.csv"
    required = [dataset, images_dir, labels_dir, calib_dir, map_path]
    if any(not p.exists() for p in required):
        stop(
            "팀 공유 데이터셋이 아직 없습니다. Drive에 shaft_sweep/dataset 폴더가 보이는지 "
            "확인하고, 없으면 다겸님께 문의하세요.",
            code=0,
        )

    images = image_files(images_dir)
    labels = sorted(p for p in labels_dir.glob("*.txt") if not p.name.startswith("._"))
    calib_images = image_files(calib_dir)
    if len(images) != 161:
        stop(f"공유 이미지가 161장이 아니라 {len(images)}장입니다. 데이터셋을 수정하지 말고 다겸님께 문의하세요.")
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    if image_stems != label_stems:
        missing = sorted(image_stems - label_stems)[:5]
        extra = sorted(label_stems - image_stems)[:5]
        stop(f"이미지와 라벨 이름이 맞지 않습니다. 누락={missing}, 초과={extra}")
    if not calib_images:
        stop("dataset/calib에 기준부품 이미지가 없습니다. 캘리브레이션 이미지를 넣어 달라고 요청하세요.")

    bad_width = []
    for p in images + calib_images:
        im = cv2.imread(str(p))
        if im is None or im.shape[1] != 2992:
            bad_width.append(p.name)
    if bad_width:
        stop(f"가로 2992px가 아니거나 읽을 수 없는 이미지가 있습니다: {bad_width[:5]}")

    frame = pd.read_csv(map_path)
    needed = {"image", "specimen_id", "is_ng", "true_mm"}
    if not needed.issubset(frame.columns):
        stop(f"specimen_map.csv 컬럼이 부족합니다. 필요한 컬럼: {sorted(needed)}")
    if len(frame) != len(images) or frame["image"].duplicated().any():
        stop("specimen_map.csv는 이미지 161장을 각각 정확히 한 번 포함해야 합니다.")
    if set(frame["image"].astype(str)) != {p.name for p in images}:
        stop("specimen_map.csv의 image 목록과 dataset/images의 파일 목록이 다릅니다.")
    if frame[list(needed)].isna().any().any():
        stop("specimen_map.csv에 빈 값이 있습니다. specimen_id/is_ng/true_mm를 모두 채워 달라고 요청하세요.")
    frame["specimen_id"] = frame["specimen_id"].astype(str)
    frame["is_ng"] = frame["is_ng"].astype(str).str.strip().str.lower().map(
        {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}
    )
    if frame["is_ng"].isna().any():
        stop("specimen_map.csv의 is_ng는 True/False 또는 1/0으로 적어야 합니다.")
    frame["true_mm"] = pd.to_numeric(frame["true_mm"], errors="coerce")
    if frame["true_mm"].isna().any():
        stop("specimen_map.csv의 true_mm에 숫자가 아닌 값이 있습니다.")
    if frame["specimen_id"].nunique() != 13:
        stop(f"시편 수가 13개가 아니라 {frame['specimen_id'].nunique()}개입니다. CSV를 확인하세요.")
    specimen_ng_counts = frame.groupby("specimen_id")["is_ng"].nunique()
    if (specimen_ng_counts != 1).any():
        stop("한 specimen_id 안에서 is_ng 값이 서로 다릅니다. CSV를 확인하세요.")
    print(f"  이미지={len(images)}, 라벨={len(labels)}, 캘리브레이션={len(calib_images)}, 시편=13 확인")
    return dataset, images_dir, calib_dir, frame, calib_images


def make_specimen_split(frame, np):
    """정렬 후 seed 고정 셔플. 정상/불량 층화, val에 불량 시편 최소 1개."""
    specimen = frame.groupby("specimen_id", as_index=False)["is_ng"].first()
    normal = sorted(specimen.loc[~specimen["is_ng"], "specimen_id"].tolist())
    ng = sorted(specimen.loc[specimen["is_ng"], "specimen_id"].tolist())
    if len(normal) != 10 or len(ng) != 3:
        stop(f"정상/불량 시편 수가 10/3이 아닙니다(현재 {len(normal)}/{len(ng)}). CSV를 확인하세요.")
    rng = np.random.default_rng(SEED)
    rng.shuffle(normal)
    rng.shuffle(ng)
    n_val = max(1, int(round((len(normal) + len(ng)) * RATIO[1])))
    n_val_ng = max(1, int(round(len(ng) * RATIO[1])))
    val = sorted(ng[:n_val_ng] + normal[: n_val - n_val_ng])
    train = sorted(set(normal + ng) - set(val))
    assert set(train).isdisjoint(val)
    assert len(train) + len(val) == 13 and any(x in set(ng) for x in val)
    return train, val


def write_data_yaml(work_dir: Path, images_dir: Path, frame, train_ids, val_ids):
    work_dir.mkdir(parents=True, exist_ok=True)
    train_set, val_set = set(train_ids), set(val_ids)
    train_paths, val_paths = [], []
    for row in frame.sort_values("image").itertuples(index=False):
        target = str((images_dir / row.image).resolve())
        if row.specimen_id in train_set:
            train_paths.append(target)
        elif row.specimen_id in val_set:
            val_paths.append(target)
        else:
            stop(f"split에 없는 시편이 발견됐습니다: {row.specimen_id}")
    (work_dir / "train.txt").write_text("\n".join(train_paths) + "\n", encoding="utf-8")
    (work_dir / "val.txt").write_text("\n".join(val_paths) + "\n", encoding="utf-8")
    yaml_text = (
        f"path: {images_dir.parent.as_posix()}\n"
        f"train: {(work_dir / 'train.txt').as_posix()}\n"
        f"val: {(work_dir / 'val.txt').as_posix()}\n"
        "names:\n  0: measurement_roi\n"
    )
    data_yaml = work_dir / "data.yaml"
    data_yaml.write_text(yaml_text, encoding="utf-8")
    return data_yaml


def is_oom(exc: BaseException) -> bool:
    text = str(exc).lower()
    return "out of memory" in text or "cuda error: out of memory" in text


def train_model(YOLO, data_yaml: Path, run_dir: Path, device):
    train_args = dict(
        data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=device, seed=SEED, deterministic=True,
        project=str(run_dir.parent), name=run_dir.name, exist_ok=True,
        optimizer="AdamW", lr0=0.001, patience=20, workers=2, plots=True,
        mosaic=0.5, mixup=0.0, copy_paste=0.0,
        degrees=0.0, shear=0.0, perspective=0.0,
        fliplr=0.0, flipud=0.0, scale=0.2, translate=0.05,
    )
    last = run_dir / "weights" / "last.pt"
    batch_used = BATCH
    start = time.perf_counter()
    try:
        if last.exists():
            print("  [resume] 드라이브에 last.pt 있음 → 이어서 학습")
            model = YOLO(str(last))
            try:
                model.train(resume=True)
            except (AssertionError, ValueError) as exc:
                # 이미 끝난 run은 resume이 안 된다. 그때는 학습을 건너뛰고 평가만 다시 한다.
                # (캘리브레이션만 고쳐서 측정 지표를 다시 뽑는 경우가 여기 해당한다.)
                if (run_dir / "weights" / "best.pt").exists() and "resume" in str(exc).lower():
                    print("  [건너뜀] 이 run은 이미 학습이 끝났습니다 → 학습 없이 평가만 진행합니다.")
                    print("           처음부터 다시 학습하려면 runs/<모델명> 폴더를 옮기거나 지우세요.")
                else:
                    raise
        else:
            model = YOLO(MODEL)
            model.train(**train_args)
    except RuntimeError as exc:
        if not is_oom(exc) or BATCH < 2:
            raise
        batch_used = max(1, BATCH // 2)
        print(f"  [OOM] GPU 메모리가 부족해 batch를 {BATCH}→{batch_used}로 낮춰 한 번 재시도합니다.")
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
        retry_last = run_dir / "weights" / "last.pt"
        if retry_last.exists():
            model = YOLO(str(retry_last))
            model.train(resume=True, batch=batch_used)
        else:
            model = YOLO(MODEL)
            model.train(**{**train_args, "batch": batch_used})
    minutes = (time.perf_counter() - start) / 60.0
    # resume/OOM 이력이 있으면 args.yaml에 남은 실제 batch가 JSON의 기준이다.
    args_yaml = run_dir / "args.yaml"
    if args_yaml.exists():
        try:
            import yaml
            saved_batch = yaml.safe_load(args_yaml.read_text(encoding="utf-8")).get("batch")
            if saved_batch is not None:
                batch_used = int(saved_batch)
        except Exception:
            pass
    best = run_dir / "weights" / "best.pt"
    if not best.exists():
        stop("학습은 끝났지만 best.pt를 찾지 못했습니다. runs 폴더와 로그를 확인하세요.")
    return YOLO(str(best)), batch_used, minutes, best


def subpixel_edge(abs_gradient, index, np):
    if index <= 0 or index >= len(abs_gradient) - 1:
        return float(index)
    g1, g2, g3 = map(float, abs_gradient[index - 1:index + 2])
    denominator = g1 - 2.0 * g2 + g3
    if abs(denominator) < 1e-12:
        return float(index)
    offset = 0.5 * (g1 - g3) / denominator
    return float(index) + float(np.clip(offset, -1.0, 1.0))


def find_outer_edge(gray_image, x, np):
    """measurement_engine.py V2의 edge 후보·선택 로직을 그대로 함수화했다."""
    strip_x1, strip_x2 = max(0, x - 2), min(gray_image.shape[1], x + 3)
    profile = np.mean(gray_image[:, strip_x1:strip_x2], axis=1).astype(np.float32)
    gradient = np.gradient(profile)
    abs_gradient = np.abs(gradient)
    h = len(profile)
    upper_end, lower_start = int(h * UPPER_END), int(h * LOWER_START)
    upper_abs, lower_abs = abs_gradient[:upper_end], abs_gradient[lower_start:]
    if len(upper_abs) == 0 or len(lower_abs) == 0:
        return None
    upper_threshold = np.percentile(upper_abs, GRAD_PERCENTILE)
    lower_threshold = np.percentile(lower_abs, GRAD_PERCENTILE)
    upper_candidates = np.where(upper_abs >= upper_threshold)[0]
    lower_candidates = np.where(lower_abs >= lower_threshold)[0] + lower_start
    candidate_pairs = []
    for top_idx in upper_candidates:
        top_gradient = float(gradient[top_idx])
        for bottom_idx in lower_candidates:
            bottom_gradient = float(gradient[bottom_idx])
            if top_gradient * bottom_gradient >= 0:
                continue
            diameter_integer = bottom_idx - top_idx
            if diameter_integer < h * MIN_DIAMETER or diameter_integer > h * MAX_DIAMETER:
                continue
            strength = abs(top_gradient) + abs(bottom_gradient)
            candidate_pairs.append((int(top_idx), int(bottom_idx), float(diameter_integer), float(strength)))
    if not candidate_pairs:
        return None
    maximum_strength = float(np.max(np.array([pair[3] for pair in candidate_pairs])))
    strong_pairs = [pair for pair in candidate_pairs if pair[3] >= maximum_strength * STRENGTH_RATIO]
    if not strong_pairs:
        strong_pairs = candidate_pairs
    top_idx, bottom_idx = map(int, max(strong_pairs, key=lambda pair: pair[2])[:2])
    top_sub = subpixel_edge(abs_gradient, top_idx, np)
    bottom_sub = subpixel_edge(abs_gradient, bottom_idx, np)
    return top_sub, bottom_sub, bottom_sub - top_sub


def blank_measurement(status: str, **updates):
    result = {
        "status": status, "yolo_conf": None,
        "box_x1": None, "box_y1": None, "box_x2": None, "box_y2": None,
        "box_w": None, "box_h": None, "n_measured": 0, "n_valid": 0,
        "measured_pixel": None, "std_pixel": None, "measured_mm": None,
    }
    result.update(updates)
    return result


def measure(image_path, model, mm_per_pixel, cv2, np) -> dict:
    """한 장 실패가 전체 평가를 중단시키지 않도록 status로 사유를 반환한다."""
    image = cv2.imread(str(image_path))
    if image is None:
        return blank_measurement("image_read_error")
    image_h, image_w = image.shape[:2]
    try:
        predictions = model.predict(source=str(image_path), conf=CONF, imgsz=IMGSZ, verbose=False)
    except Exception:
        return blank_measurement("no_roi")
    candidates = []
    for prediction in predictions:
        if prediction.boxes is not None:
            for box in prediction.boxes:
                candidates.append((float(box.conf[0]), box))
    if not candidates:
        return blank_measurement("no_roi")
    best_conf, best_box = max(candidates, key=lambda item: item[0])
    x1, y1, x2, y2 = best_box.xyxy[0].cpu().numpy().astype(int)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(image_w, x2), min(image_h, y2)
    base = dict(
        yolo_conf=best_conf, box_x1=int(x1), box_y1=int(y1),
        box_x2=int(x2), box_y2=int(y2), box_w=int(x2 - x1), box_h=int(y2 - y1),
    )
    box_h = y2 - y1
    vertical_margin = int(box_h * ROI_VERTICAL_MARGIN)
    measure_y1, measure_y2 = max(0, y1 - vertical_margin), min(image_h, y2 + vertical_margin)
    roi = image[measure_y1:measure_y2, x1:x2].copy()  # 가로 고정, 세로만 40% 확장
    if roi.size == 0:
        return blank_measurement("insufficient_edges", **base)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, roi_w = blur.shape
    x_positions = np.linspace(int(roi_w * X_START), int(roi_w * X_END), N_LINES).astype(int)
    diameters = []
    for x in x_positions:
        edge = find_outer_edge(blur, x, np)
        if edge is not None:
            diameters.append(edge[2])
    if len(diameters) < MIN_RAW_EDGES:
        return blank_measurement("insufficient_edges", n_measured=len(diameters), **base)
    diameters = np.array(diameters, dtype=np.float64)
    raw_median = float(np.median(diameters))
    deviation = np.abs(diameters - raw_median)
    mad = float(np.median(deviation))
    if mad > 0:
        valid_mask = deviation <= MAD_K * 1.4826 * mad
    else:
        valid_mask = np.ones(len(diameters), dtype=bool)
    valid = diameters[valid_mask]
    if len(valid) < MIN_VALID_EDGES:
        return blank_measurement(
            "outlier_reject", n_measured=len(diameters), n_valid=len(valid), **base
        )
    measured_pixel = float(np.median(valid))
    return blank_measurement(
        "ok", n_measured=len(diameters), n_valid=len(valid),
        measured_pixel=measured_pixel, std_pixel=float(np.std(valid)),
        measured_mm=float(measured_pixel * mm_per_pixel) if mm_per_pixel is not None else None,
        **base,
    )


def scalar(value):
    try:
        return float(value.item())
    except Exception:
        try:
            return float(value)
        except Exception:
            return None


def detection_metrics(model, data_yaml: Path, device):
    val = model.val(data=str(data_yaml), split="val", imgsz=IMGSZ, device=device, verbose=False)
    rd = getattr(val, "results_dict", {}) or {}
    speed = getattr(val, "speed", {}) or {}
    return {
        "map50": scalar(rd.get("metrics/mAP50(B)", getattr(getattr(val, "box", None), "map50", None))),
        "map50_95": scalar(rd.get("metrics/mAP50-95(B)", getattr(getattr(val, "box", None), "map", None))),
        "precision": scalar(rd.get("metrics/precision(B)", getattr(getattr(val, "box", None), "mp", None))),
        "recall": scalar(rd.get("metrics/recall(B)", getattr(getattr(val, "box", None), "mr", None))),
        "infer_ms": scalar(speed.get("inference")),
    }


def model_size(model):
    params = int(sum(p.numel() for p in model.model.parameters()))
    gflops = None
    try:
        info = model.info(verbose=False)
        if isinstance(info, tuple) and len(info) >= 4:
            gflops = scalar(info[3])
    except Exception:
        pass
    return params, gflops


def training_csv_summary(run_dir: Path, pd):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return None, None
    table = pd.read_csv(results_csv)
    table.columns = [c.strip() for c in table.columns]
    total_minutes = None
    if "time" in table.columns and len(table):
        # Ultralytics results.csv의 time은 run 시작 후 누적 초다.
        total_minutes = float(pd.to_numeric(table["time"], errors="coerce").max()) / 60.0
    fitness_cols = [c for c in table.columns if c == "fitness"]
    if fitness_cols:
        return int(table[fitness_cols[0]].idxmax()) + 1, total_minutes
    map_cols = [c for c in table.columns if "mAP50-95" in c]
    epoch = int(table[map_cols[0]].idxmax()) + 1 if map_cols else int(len(table))
    return epoch, total_minutes


def mean_group_std(frame, value_col, pd):
    good = frame.dropna(subset=[value_col])
    if good.empty:
        return None
    values = good.groupby("specimen_id")[value_col].std(ddof=0)
    return float(values.mean()) if len(values) else None


def evaluate_measurement(model, frame, images_dir, calib_images, cv2, np, pd):
    print("[6/7] 모델별 캘리브레이션과 val 측정을 실행합니다.")
    calib_rows = []
    for path in calib_images:
        row = {"image": path.name, **measure(path, model, None, cv2, np)}
        calib_rows.append(row)
    pixels = [r["measured_pixel"] for r in calib_rows if r["status"] == "ok"]
    if not pixels:
        stop("캘리브레이션 이미지에서 유효한 측정값이 하나도 나오지 않았습니다. calib 이미지와 모델을 확인하세요.")
    # ── 가드 ①: 캘리브레이션 표본 수 ─────────────────────────────
    if len(pixels) < MIN_CALIB_IMAGES:
        stop(
            f"캘리브레이션에 쓸 수 있는 이미지가 {len(pixels)}장뿐입니다 (최소 {MIN_CALIB_IMAGES}장).\n"
            "  이 설계는 calib 이미지들의 '중앙값'을 기준 픽셀로 씁니다.\n"
            "  1~2장이면 그 사진의 박스가 한 번 흔들릴 때 모든 측정값이 통째로 어긋납니다.\n"
            "  → dataset/calib/에 같은 시편을 같은 세션에서 찍은 사진을 3장 이상 넣어 주세요."
        )
    calib_array = np.array(pixels, dtype=float)
    calibration_pixel = float(np.median(calib_array))
    calib_spread_pct = float((calib_array.max() - calib_array.min()) / calibration_pixel * 100.0)
    # ── 가드 ②: calib 이미지끼리의 산포 ──────────────────────────
    if calib_spread_pct > CALIB_SPREAD_MAX_PCT:
        stop(
            f"캘리브레이션 이미지끼리 측정 픽셀이 너무 벌어집니다: {calib_array.min():.2f} ~ {calib_array.max():.2f}px "
            f"(폭 {calib_spread_pct:.1f}%, 허용 {CALIB_SPREAD_MAX_PCT:.1f}%).\n"
            "  같은 시편을 같은 세션에서 찍은 사진인지, 박스가 매번 같은 구간에 앉는지 확인하세요.\n"
            "  (서로 다른 시편이 섞여 있으면 이 경고가 납니다.)"
        )
    mm_per_pixel = float(CALIBRATION_MM / calibration_pixel)
    print(f"  캘리브레이션: {len(pixels)}장, 중앙값 {calibration_pixel:.2f}px "
          f"(폭 {calib_spread_pct:.2f}%), mm_per_pixel={mm_per_pixel:.6f}")

    # 파일명이 겹치거나 같은 실제 파일인 경우 모두 오차 평가에서 제외한다.
    calib_names = {p.name for p in calib_images}
    val_rows = []
    for meta in frame.loc[frame["split"] == "val"].sort_values("image").itertuples(index=False):
        if meta.image in calib_names:
            continue
        measured = measure(images_dir / meta.image, model, mm_per_pixel, cv2, np)
        measured.update(
            image=meta.image, specimen_id=meta.specimen_id,
            is_ng=bool(meta.is_ng), true_mm=float(meta.true_mm), split="val",
        )
        if measured["status"] == "ok":
            measured["error_mm"] = measured["measured_mm"] - measured["true_mm"]
            measured["predicted_ng"] = not (SPEC_LOWER <= measured["measured_mm"] <= SPEC_UPPER)
        else:
            measured["error_mm"], measured["predicted_ng"] = None, None
        val_rows.append(measured)

    # ── 가드 ③: calib 배율이 본 이미지와 같은 세션에서 나온 것인가 ──
    # measured_pixel은 mm_per_pixel과 무관하게 구해지므로 여기서 바로 비교할 수 있다.
    val_pixels = [
        r["measured_pixel"] for r in val_rows
        if r["status"] == "ok" and r["measured_pixel"] is not None
    ]
    if not val_pixels:
        stop("val 이미지에서 유효한 측정값이 하나도 나오지 않았습니다. 모델과 라벨을 확인하세요.")
    val_pixel_median = float(np.median(np.array(val_pixels, dtype=float)))
    scale_gap_pct = float(abs(calibration_pixel - val_pixel_median) / val_pixel_median * 100.0)
    if scale_gap_pct > CALIB_SCALE_MAX_PCT:
        stop(
            "캘리브레이션 배율이 측정 이미지와 맞지 않습니다.\n"
            f"  calib 측정 픽셀(중앙값)      = {calibration_pixel:.2f}px\n"
            f"  본 이미지 측정 픽셀(중앙값)  = {val_pixel_median:.2f}px\n"
            f"  차이 = {scale_gap_pct:.1f}%  (허용 {CALIB_SCALE_MAX_PCT:.0f}%)\n"
            f"  이대로 두면 측정값이 전부 약 {val_pixel_median / calibration_pixel:.3f}배로 어긋납니다.\n"
            "  → dataset/calib/ 이미지가 측정 이미지와 '다른 촬영 세션'(카메라 거리가 다름)일 가능성이 큽니다.\n"
            "     해상도가 같아도 촬영 거리가 다르면 부품이 차지하는 픽셀 수가 달라집니다.\n"
            "     161장과 같은 세션에서 찍은 사진으로 교체하고, CALIBRATION_MM도 그 시편의 실측값으로 바꾸세요."
        )
    print(f"  배율 점검 OK: calib {calibration_pixel:.2f}px vs 본 이미지 {val_pixel_median:.2f}px "
          f"(차이 {scale_gap_pct:.2f}%)")

    result_frame = pd.DataFrame(val_rows)
    ok = result_frame.loc[result_frame["status"] == "ok"].copy()
    errors = ok["error_mm"].astype(float).to_numpy() if len(ok) else np.array([])
    status_counts = Counter(result_frame["status"].tolist())
    # OK/NG는 반복 사진 수가 많은 시편에 가중치가 쏠리지 않도록 시편 median으로 판정한다.
    specimen_eval = (
        ok.groupby("specimen_id", as_index=False)
        .agg(measured_mm=("measured_mm", "median"), is_ng=("is_ng", "first"))
        if len(ok) else pd.DataFrame(columns=["specimen_id", "measured_mm", "is_ng"])
    )
    specimen_eval["predicted_ng"] = ~specimen_eval["measured_mm"].between(SPEC_LOWER, SPEC_UPPER)
    measure_metrics = {
        "bias_um": float(np.mean(errors) * 1000.0) if len(errors) else None,
        "mae_um": float(np.mean(np.abs(errors)) * 1000.0) if len(errors) else None,
        "rmse_um": float(np.sqrt(np.mean(np.square(errors))) * 1000.0) if len(errors) else None,
        "repeat_sigma_um": (
            mean_group_std(ok, "measured_mm", pd) * 1000.0 if len(ok) else None
        ),
        "box_h_std_px": mean_group_std(ok, "box_h", pd) if len(ok) else None,
        "box_y1_std_px": mean_group_std(ok, "box_y1", pd) if len(ok) else None,
        "box_y2_std_px": mean_group_std(ok, "box_y2", pd) if len(ok) else None,
        "ng_detected": int((specimen_eval["is_ng"] & specimen_eval["predicted_ng"]).sum()),
        "false_alarm": int((~specimen_eval["is_ng"] & specimen_eval["predicted_ng"]).sum()),
        "n_ng_specimens": int(specimen_eval["is_ng"].sum()),
        "n_normal_specimens": int((~specimen_eval["is_ng"]).sum()),
        "n_eval_images": int(len(result_frame)),
        "n_eval_specimens": int(result_frame["specimen_id"].nunique()) if len(result_frame) else 0,
    }
    robust = {
        "n_ok": int(status_counts.get("ok", 0)),
        "n_no_roi": int(status_counts.get("no_roi", 0)),
        "n_insufficient_edges": int(status_counts.get("insufficient_edges", 0)),
        "n_outlier_reject": int(status_counts.get("outlier_reject", 0)),
    }
    calib_diag = {
        "n_used": len(pixels),
        "spread_pct": calib_spread_pct,
        "val_pixel_median": val_pixel_median,
        "scale_gap_pct": scale_gap_pct,
        "guard_pct": CALIB_SCALE_MAX_PCT,
    }
    return calibration_pixel, mm_per_pixel, measure_metrics, robust, result_frame, calib_rows, calib_diag


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if value is None:
        return None
    try:
        if math.isnan(float(value)) or math.isinf(float(value)):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, "item"):
        return value.item()
    return value


def main():
    if MODEL not in ALLOWED_MODELS:
        stop(f"MODEL 값이 비교 대상 6종에 없습니다: {MODEL}")
    cv2, np, pd, torch, ultralytics, YOLO, device, gpu = prepare_environment()
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    root = Path(DRIVE)
    (root / "logs").mkdir(parents=True, exist_ok=True)
    log_path = root / "logs" / f"{Path(MODEL).stem}.log"
    logging.basicConfig(filename=log_path, level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    logging.info("실행 시작: model=%s ultralytics=%s gpu=%s", MODEL, ultralytics.__version__, gpu)
    print("[2/7] 공유 데이터셋을 검사합니다.")
    dataset, images_dir, calib_dir, frame, calib_images = validate_dataset(root, cv2, pd)
    print("[3/7] 시편 단위 고정 split을 만듭니다.")
    train_ids, val_ids = make_specimen_split(frame, np)
    frame["split"] = frame["specimen_id"].map(lambda x: "train" if x in set(train_ids) else "val")
    run_name = Path(MODEL).stem
    run_dir = root / "runs" / run_name
    data_yaml = write_data_yaml(run_dir / "split", images_dir, frame, train_ids, val_ids)
    print(f"  train 시편={train_ids}")
    print(f"  val 시편={val_ids}")

    print(f"[4/7] {MODEL} 학습을 시작합니다.")
    model, batch_used, train_minutes, best_path = train_model(YOLO, data_yaml, run_dir, device)
    print("[5/7] val 탐지 지표를 계산합니다.")
    detect = detection_metrics(model, data_yaml, device)
    params, gflops = model_size(model)
    epoch, csv_train_minutes = training_csv_summary(run_dir, pd)
    if csv_train_minutes is not None:
        train_minutes = csv_train_minutes
    calibration_pixel, mm_per_pixel, measure_metrics, robust, per_image, calib_rows, calib_diag = evaluate_measurement(
        model, frame, images_dir, calib_images, cv2, np, pd
    )

    print("[7/7] 결과 파일을 저장합니다.")
    results_dir = root / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    per_image_path = results_dir / f"per_image_{run_name}.csv"
    per_image.to_csv(per_image_path, index=False, encoding="utf-8-sig")
    executor = getpass.getuser()
    executed_at = datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")
    result = {
        "model": MODEL,
        "ultralytics_version": ultralytics.__version__,
        "gpu": gpu,
        "executor": executor,
        "executed_at": executed_at,
        "실행자": executor,
        "실행시각": executed_at,
        "sample": {"n_images": 161, "n_specimens": 13, "n_val_images": int(len(per_image))},
        "train": {
            "epochs": EPOCHS, "batch_used": batch_used, "best_epoch": epoch,
            "train_minutes": train_minutes, "params": params, "gflops": gflops,
            "best_weights": str(best_path),
        },
        "detect": detect,
        "calib": {
            "calibration_mm": CALIBRATION_MM, "calibration_pixel": calibration_pixel,
            "mm_per_pixel": mm_per_pixel, "n_images": len(calib_images), "n_ok": len([r for r in calib_rows if r["status"] == "ok"]),
            "excluded_from_error_evaluation": True,
            "n_used_for_median": calib_diag["n_used"],
            "calib_spread_pct": calib_diag["spread_pct"],
            "val_pixel_median": calib_diag["val_pixel_median"],
            "scale_gap_pct": calib_diag["scale_gap_pct"],
            "scale_guard_pct": calib_diag["guard_pct"],
        },
        "measure": measure_metrics,
        "robust": robust,
        "split": {"seed": SEED, "train_specimens": train_ids, "val_specimens": val_ids},
        "limitations": [
            "라벨은 기존 best.pt의 pre-annotation을 사람이 검수한 것으로, 기존 박스 규칙 재현 성능을 비교한다.",
            "시편 13개, 이미지 161장의 작은 표본이다.",
        ],
    }
    result_path = results_dir / f"result_{run_name}.json"
    result_path.write_text(json.dumps(json_safe(result), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    def fmt(value, digits=3):
        return "계산 불가" if value is None else f"{value:.{digits}f}"
    print("\n========== 내 모델 결과 요약 ==========")
    print(f"1. 모델 / split : {MODEL} / train {len(train_ids)}시편, val {len(val_ids)}시편")
    print(f"2. 탐지 성능     : mAP50-95={fmt(detect['map50_95'], 4)}, mAP50={fmt(detect['map50'], 4)}")
    print(f"3. 측정 정확도   : bias={fmt(measure_metrics['bias_um'])}µm, MAE={fmt(measure_metrics['mae_um'])}µm")
    print(f"4. 측정 재현성   : σ={fmt(measure_metrics['repeat_sigma_um'])}µm (최종 결론 지표)")
    print(f"5. 배율 점검     : calib {calibration_pixel:.2f}px vs 본 이미지 {calib_diag['val_pixel_median']:.2f}px "
          f"(차이 {calib_diag['scale_gap_pct']:.2f}%, 허용 {calib_diag['guard_pct']:.0f}%)")
    print(f"6. 저장 완료     : {result_path}")


if __name__ == "__main__":
    try:
        main()
    except SystemExit:
        raise
    except Exception as exc:
        logging.exception("실행 중 예외")
        print("\n[오류] 실행 중 예상하지 못한 문제가 생겼습니다.")
        print(f"원인: {type(exc).__name__}: {exc}")
        print("해결: 위 메시지와 logs 폴더의 로그를 다겸님께 전달하세요.")
        traceback.print_exc()
        raise SystemExit(1)
